In [19]:
import pandas as pd
import os
from zipfile import ZipFile
import requests

# ----------------------------
# 1. Download & Extract MovieLens 1M
# ----------------------------
dataset_url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
dataset_zip = "ml-1m.zip"
extracted_path = "ml-1m"

if not os.path.exists(extracted_path):
    print("Downloading MovieLens 1M dataset...")
    r = requests.get(dataset_url)
    with open(dataset_zip, "wb") as f:
        f.write(r.content)
    with ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Download and extraction done.")

# ----------------------------
# 2. Load all files
# ----------------------------
ratings_file = os.path.join(extracted_path, "ratings.dat")
movies_file = os.path.join(extracted_path, "movies.dat")
users_file  = os.path.join(extracted_path, "users.dat")

ratings = pd.read_csv(
    ratings_file, sep="::", engine="python",
    names=["UserID","MovieID","Rating","Timestamp"]
)
movies = pd.read_csv(
    movies_file, sep="::", engine="python",
    names=["MovieID","Title","Genres"], encoding="latin-1"
)
users = pd.read_csv(
    users_file, sep="::", engine="python",
    names=["UserID","Gender","Age","Occupation","Zip-code"]
)

# ----------------------------
# 3. Convert timestamp
# ----------------------------
ratings['Timestamp'] = pd.to_datetime(ratings['Timestamp'], unit='s')

# ----------------------------
# 4. Check for duplicates
# ----------------------------
print("Duplicate rows:")
print(f"Ratings: {ratings.duplicated().sum()}")
print(f"Movies:  {movies.duplicated().sum()}")
print(f"Users:   {users.duplicated().sum()}")

# Drop duplicates
ratings = ratings.drop_duplicates().reset_index(drop=True)
movies  = movies.drop_duplicates().reset_index(drop=True)
users   = users.drop_duplicates().reset_index(drop=True)

# ----------------------------
# 5. Check for missing values in detail
# ----------------------------
def report_missing(df, name):
    total_missing = df.isnull().sum().sum()
    if total_missing == 0:
        print(f"No missing values in {name}.")
    else:
        print(f"Missing values in {name}:")
        for col in df.columns:
            missing_count = df[col].isnull().sum()
            if missing_count > 0:
                print(f" - Column '{col}': {missing_count} missing")
        print("Rows with missing values:")
        print(df[df.isnull().any(axis=1)])

report_missing(ratings, "ratings")
report_missing(movies, "movies")
report_missing(users, "users")

# ----------------------------
# 6. Filter ratings to May → Dec 2000
# ----------------------------
start_date = '2000-05-01'
end_date   = '2000-12-31'
ratings = ratings[(ratings['Timestamp'] >= start_date) & (ratings['Timestamp'] <= end_date)].reset_index(drop=True)
print("\nRatings after filtering May → Dec 2000:", ratings.shape)


Duplicate rows:
Ratings: 0
Movies:  0
Users:   0
No missing values in ratings.
No missing values in movies.
No missing values in users.

Ratings after filtering May → Dec 2000: (891189, 4)


In [20]:
# ----------------------------
# Create monthly slices
# ----------------------------
monthly_bins = pd.date_range(start=start_date, end='2001-01-01', freq='MS')
ratings['SliceNum'] = pd.cut(ratings['Timestamp'], bins=monthly_bins, right=False).cat.codes + 1

# ----------------------------
# Inspect each slice
# ----------------------------
slice_summary = []

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    num_ratings = slice_df.shape[0]
    num_movies  = slice_df['MovieID'].nunique()
    num_users   = slice_df['UserID'].nunique()
    slice_summary.append({
        'SliceNum': s,
        'Ratings': num_ratings,
        'UniqueMovies': num_movies,
        'UniqueUsers': num_users
    })

slice_summary_df = pd.DataFrame(slice_summary)
print(slice_summary_df)


   SliceNum  Ratings  UniqueMovies  UniqueUsers
0         1    67437          2920          486
1         2    54486          2913          508
2         3    90334          3135          778
3         4   182109          3298         1310
4         5    52421          3083          576
5         6    42294          2993          500
6         7   290793          3552         2357
7         8   111315          3331         1215


In [21]:
import os
import pandas as pd

# ----------------------------
# 1. Create folder for slice metadata
# ----------------------------
os.makedirs("slice_metadata", exist_ok=True)

# ----------------------------
# 2. Split movies into popularity tiers per slice
# ----------------------------
high_pct, mid_pct, low_pct = 0.2, 0.3, 0.5  # 20/30/50 split

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    movie_counts = slice_df.groupby('MovieID')['Rating'].count().sort_values(ascending=False)
    total_movies = len(movie_counts)
    
    # Determine cutoff indices
    high_cut  = int(total_movies * high_pct)
    mid_cut   = high_cut + int(total_movies * mid_pct)
    
    # Assign tiers
    movie_tiers = pd.DataFrame({'MovieID': movie_counts.index})
    movie_tiers['PopTier'] = 'low'  # default
    movie_tiers.loc[:high_cut-1, 'PopTier'] = 'high'
    movie_tiers.loc[high_cut:mid_cut-1, 'PopTier'] = 'medium'
    
    # Save slice metadata
    movie_tiers.to_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv", index=False)
    print(f"Slice {s}: {total_movies} movies → High:{high_cut}, Medium:{mid_cut-high_cut}, Low:{total_movies-mid_cut}")


Slice 1: 2920 movies → High:584, Medium:876, Low:1460
Slice 2: 2913 movies → High:582, Medium:873, Low:1458
Slice 3: 3135 movies → High:627, Medium:940, Low:1568
Slice 4: 3298 movies → High:659, Medium:989, Low:1650
Slice 5: 3083 movies → High:616, Medium:924, Low:1543
Slice 6: 2993 movies → High:598, Medium:897, Low:1498
Slice 7: 3552 movies → High:710, Medium:1065, Low:1777
Slice 8: 3331 movies → High:666, Medium:999, Low:1666


In [22]:
# ----------------------------
# User type assignment per slice
# ----------------------------
import os
import pandas as pd

slice_nums = sorted(ratings['SliceNum'].unique())
os.makedirs("slice_metadata", exist_ok=True)

threshold = 0.7  # HighProportion threshold to classify mainstream

for s in slice_nums:
    slice_df = ratings[ratings['SliceNum'] == s].copy()
    if slice_df.empty: 
        continue

    # Load movie popularity tiers for this slice
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv")
    movie_tier_map = dict(zip(movie_tiers.MovieID, movie_tiers.PopTier))

    # Map each rating to movie tier
    slice_df['Tier'] = slice_df['MovieID'].map(movie_tier_map)

    # Count number of ratings per tier per user
    tier_counts = slice_df.groupby(['UserID', 'Tier']).size().unstack(fill_value=0)
    # Ensure all columns exist
    for col in ['high', 'medium', 'low']:
        if col not in tier_counts.columns:
            tier_counts[col] = 0

    # Compute HighProportion
    tier_counts['Total'] = tier_counts['high'] + tier_counts['medium'] + tier_counts['low']
    tier_counts['HighProportion'] = tier_counts['high'] / tier_counts['Total']

    # Assign user type
    tier_counts['UserType'] = tier_counts['HighProportion'].apply(
        lambda x: 'mainstream' if x >= threshold else 'niche'
    )

    # Save result
    user_type_df = tier_counts[['UserType']].reset_index()
    user_type_df.to_csv(f"slice_metadata/user_labels_slice_{s}.csv", index=False)
    
    print(f"Slice {s}: Total Users = {len(user_type_df)}, Mainstream = {sum(user_type_df.UserType=='mainstream')}, Niche = {sum(user_type_df.UserType=='niche')}")


Slice 1: Total Users = 486, Mainstream = 233, Niche = 253
Slice 2: Total Users = 508, Mainstream = 239, Niche = 269
Slice 3: Total Users = 778, Mainstream = 332, Niche = 446
Slice 4: Total Users = 1310, Mainstream = 598, Niche = 712
Slice 5: Total Users = 576, Mainstream = 276, Niche = 300
Slice 6: Total Users = 500, Mainstream = 223, Niche = 277
Slice 7: Total Users = 2357, Mainstream = 1420, Niche = 937
Slice 8: Total Users = 1215, Mainstream = 596, Niche = 619


In [23]:
import pandas as pd
import os
import torch

# ----------------------------
# 1. KG construction per slice with relation types and mappings
# ----------------------------
slice_nums = sorted(ratings['SliceNum'].unique())
os.makedirs("slice_metadata/kg", exist_ok=True)

# Define relation types
REL_USER_MOVIE = 0
REL_MOVIE_GENRE = 1
REL_MOVIE_POPTIER = 2
REL_USER_USERTYPE = 3

for s in slice_nums:
    slice_df = ratings[ratings['SliceNum'] == s].copy()
    if slice_df.empty:
        continue
    
    print(f"\nConstructing KG for slice {s}...")

    # Load slice-specific metadata
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv")
    user_types  = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv")
    
    # Extract unique nodes
    users = slice_df['UserID'].unique()
    movies_slice = slice_df['MovieID'].unique()
    genres = set()
    for g in movies['Genres']:
        genres.update(g.split('|'))
    genres = list(genres)
    
    pop_tiers = ['high', 'medium', 'low']
    utypes = ['mainstream', 'niche']
    
    # ----------------------------
    # Slice-local IDs
    user2nid = {u: i for i, u in enumerate(users)}
    movie2nid = {m: i + len(users) for i, m in enumerate(movies_slice)}
    genre2nid = {g: i + len(users) + len(movies_slice) for i, g in enumerate(genres)}
    pop2nid = {p: i + len(users) + len(movies_slice) + len(genres) for i, p in enumerate(pop_tiers)}
    utype2nid = {ut: i + len(users) + len(movies_slice) + len(genres) + len(pop_tiers) for i, ut in enumerate(utypes)}
    
    # Save mapping tables for hybrid warm-start later
    pd.DataFrame(list(user2nid.items()), columns=['UserID','NodeID']).to_csv(f"slice_metadata/kg/user_map_slice_{s}.csv", index=False)
    pd.DataFrame(list(movie2nid.items()), columns=['MovieID','NodeID']).to_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv", index=False)
    
    # ----------------------------
    # Build edges with relation types
    edges = []

    # User ↔ Movie
    for row in slice_df.itertuples():
        uid, mid = user2nid[row.UserID], movie2nid[row.MovieID]
        edges.append((uid, mid, REL_USER_MOVIE))
        edges.append((mid, uid, REL_USER_MOVIE))  # bi-directional

    # Movie ↔ Genre
    for row in movies.itertuples():
        if row.MovieID not in movie2nid:
            continue
        mid = movie2nid[row.MovieID]
        for g in row.Genres.split('|'):
            edges.append((mid, genre2nid[g], REL_MOVIE_GENRE))
            edges.append((genre2nid[g], mid, REL_MOVIE_GENRE))

    # Movie ↔ PopTier
    for row in movie_tiers.itertuples():
        mid = movie2nid[row.MovieID]
        edges.append((mid, pop2nid[row.PopTier], REL_MOVIE_POPTIER))
        edges.append((pop2nid[row.PopTier], mid, REL_MOVIE_POPTIER))

    # User ↔ UserType
    for row in user_types.itertuples():
        uid = user2nid[row.UserID]
        edges.append((uid, utype2nid[row.UserType], REL_USER_USERTYPE))
        edges.append((utype2nid[row.UserType], uid, REL_USER_USERTYPE))
    
    # Save edges with relation types
    edges_df = pd.DataFrame(edges, columns=['Src','Dst','RelType'])
    edges_df.to_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv", index=False)
    
    print(f"Slice {s} KG: {len(edges_df)} edges, {len(user2nid)} users, {len(movie2nid)} movies, {len(genre2nid)} genres, {len(pop2nid)} pop tiers, {len(utype2nid)} user types")



Constructing KG for slice 1...
Slice 1 KG: 151890 edges, 486 users, 2920 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 2...
Slice 2 KG: 126078 edges, 508 users, 2913 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 3...
Slice 3 KG: 199352 edges, 778 users, 3135 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 4...
Slice 4 KG: 384750 edges, 1310 users, 3298 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 5...
Slice 5 KG: 122904 edges, 576 users, 3083 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 6...
Slice 6 KG: 102034 edges, 500 users, 2993 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 7...
Slice 7 KG: 605404 edges, 2357 users, 3552 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 8...
Slice 8 KG: 243150 edges, 1215 users, 3331 movies, 18 genres, 3 pop tiers, 2 user types


PROJECT README
Temporal Knowledge Graphs for Recommender Systems
================================================

1. What this project is about
-----------------------------

This project studies how user preferences and item popularity change over time,
and how recommender models can adapt to those changes instead of treating data as static.

Most recommender systems:
- Merge all interactions into one dataset
- Assume user behavior and item popularity are fixed
- Ignore temporal structure

That is unrealistic.

In this project, we:
- Split user–item interactions into time slices
- Build a separate Knowledge Graph (KG) for each slice
- Train models slice by slice
- Compare a baseline recommender with a more expressive final model

The goal is to evaluate whether modeling structure and time actually improves recommendations.


2. Why use a Knowledge Graph
----------------------------

A standard user–item matrix only captures:
“User A interacted with Movie B”.

A Knowledge Graph allows us to represent richer structure:
- Users have behavioral types (mainstream vs niche)
- Movies have genres
- Movies have popularity levels
- All of these can change over time

By encoding interactions as a graph:
- Simple graph models can be used as baselines
- Relation-aware models can exploit edge types
- The same data supports multiple model families


3. Why time slicing
-------------------

User behavior in May 2000 is not the same as in December 2000.

Instead of building one global graph, we:
- Split ratings into monthly time slices
- Build one KG per slice
- Allow users and movies to appear, disappear, or change context

This enables:
- Temporal evaluation
- Embedding warm-starting
- Studying preference drift


4. Models trained later
-----------------------

This preprocessing supports two categories of models:

1. Baseline model
   - Treats all edges as the same
   - Uses only graph structure
   - Ignores relation types

2. Final model
   - Uses relation types
   - Distinguishes different kinds of edges
   - Exploits richer KG structure

Both models use the same preprocessed data.


5. Dataset
----------

- MovieLens 1M
- Ratings filtered to May 2000 – December 2000
- Monthly time slices


6. Outputs per time slice
------------------------

For each slice, the following are created:
- Movie popularity tiers
- User behavioral types
- Slice-specific Knowledge Graph
- Node ID mappings for temporal alignment

All outputs are stored as CSV files.


7. Directory structure
----------------------

.
├── slice_metadata/
│   ├── movie_pop_tiers_slice_1.csv
│   ├── user_labels_slice_1.csv
│   ├── ...
│   └── kg/
│       ├── kg_edges_slice_1.csv
│       ├── user_map_slice_1.csv
│       ├── movie_map_slice_1.csv
│       └── ...


8. Movie popularity tiers
-------------------------

Files:
slice_metadata/movie_pop_tiers_slice_{s}.csv

Description:
- Movies are ranked by number of ratings within a slice
- Assigned to popularity tiers:
  - high
  - medium
  - low

These tiers capture short-term popularity, not global popularity.


9. User behavioral types
------------------------

Files:
slice_metadata/user_labels_slice_{s}.csv

Description:
- Users are labeled based on what they consume in that slice
- Users who mostly rate popular movies → mainstream
- Others → niche

User types can change across slices.


10. Knowledge Graphs
--------------------

Each slice has its own Knowledge Graph.


10.1 Edge list
--------------

Files:
slice_metadata/kg/kg_edges_slice_{s}.csv

Columns:
- Src     : source node ID
- Dst     : destination node ID
- RelType : relation type ID

All edges are stored as bidirectional.


10.2 Relation types
-------------------

Relation ID meanings:

0 → User ↔ Movie  
1 → Movie ↔ Genre  
2 → Movie ↔ Popularity Tier  
3 → User ↔ User Type  

Baseline models ignore RelType.
Final models use RelType.


11. Node IDs
------------

Node IDs are:
- Integers
- Unique within a slice
- Assigned in non-overlapping ranges by node type

This guarantees that users, movies, genres, tiers, and user types
cannot be confused with each other.


12. Node ID mappings (important)
--------------------------------

Files:
- slice_metadata/kg/user_map_slice_{s}.csv
- slice_metadata/kg/movie_map_slice_{s}.csv

These map original UserID / MovieID to slice-local node IDs.

They are required for:
- Matching entities across slices
- Warm-starting embeddings
- Temporal stabilization during training


13. What is NOT stored
----------------------

- No trained models
- No tensors
- No pickled Python objects
- No framework-specific formats

Everything is reconstructed from CSVs in training notebooks.


14. How a new training session should start
-------------------------------------------

A fresh session should:
1. Select a slice
2. Load KG edges and mappings
3. Convert data to tensors as needed
4. Train baseline and final models
5. Optionally align embeddings across slices


15. Summary
-----------

This project builds time-aware Knowledge Graphs from MovieLens data
to compare a baseline recommender against a relation-aware model
while explicitly modeling temporal dynamics.


K = 10

In [24]:
import os
# CRITICAL FIX: This must be set BEFORE importing torch
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (STAY UNCHANGED)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=10):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj)
        num_users = len(user_map)
        final_u = final_emb[:num_users]
        final_i = final_emb[num_users : num_users + len(item_map)]
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        scores = torch.matmul(final_u[test_u_tensor], final_i.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                seen_nodes = [item_map[m] for m in train_interactions[u_id] if m in item_map]
                if seen_nodes:
                    scores[idx, seen_nodes] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        item_counts = np.zeros(len(item_map))
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            recs = top_indices[idx]
            hits = len(set(recs) & truth)
            recall_arr.append(hits / len(truth))
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(recs):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for r in recs:
                item_counts[r] += 1
                tier = movie_tiers.get(node_to_m_id[r], 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) + abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) + abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(item_counts)
        
        return {'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp}, len(test_u_nodes)

# ----------------------------
# 3. Model Logic (PinSage)
# ----------------------------
class PinSage(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, n_layers=2):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.emb.weight)
        self.layers = nn.ModuleList([nn.Linear(embedding_dim * 2, embedding_dim) for _ in range(n_layers)])
        self.activation = nn.ReLU()

    def forward(self, adj):
        h = self.emb.weight
        for layer in self.layers:
            neighbor_h = torch.sparse.mm(adj, h)
            h_combined = torch.cat([h, neighbor_h], dim=1)
            h = self.activation(layer(h_combined))
            h = nn.functional.normalize(h, p=2, dim=1)
        return h

def build_adjacency(u_list, i_list, kg_df, u_map, i_map, ent_map, num_neighbors=30, num_walks=20, walk_length=5):
    n_u, n_i, n_e = len(u_map), len(i_map), len(ent_map)
    total_nodes = n_u + n_i + n_e
    rows = np.concatenate([u_list, i_list + n_u])
    cols = np.concatenate([i_list + n_u, u_list])
    
    if not kg_df.empty:
        h_nodes = kg_df['head'].map({**i_map, **ent_map}).values
        t_nodes = kg_df['tail'].map({**i_map, **ent_map}).values
        mask = ~np.isnan(h_nodes) & ~np.isnan(t_nodes)
        h_idx = h_nodes[mask].astype(int) + n_u
        t_idx = t_nodes[mask].astype(int) + n_u
        rows = np.concatenate([rows, h_idx, t_idx])
        cols = np.concatenate([cols, t_idx, h_idx])

    adj_mat = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(total_nodes, total_nodes))
    new_rows, new_cols, new_data = [], [], []

    for node_idx in range(total_nodes):
        visit_counts = {}
        for _ in range(num_walks):
            curr = node_idx
            for _ in range(walk_length):
                neighbors = adj_mat.indices[adj_mat.indptr[curr]:adj_mat.indptr[curr+1]]
                if len(neighbors) == 0: break
                curr = np.random.choice(neighbors)
                visit_counts[curr] = visit_counts.get(curr, 0) + 1
        if not visit_counts: continue
        unique_neighbors = list(visit_counts.keys()); counts = np.array(list(visit_counts.values()), dtype=float)
        importance_probs = counts / counts.sum()
        if len(unique_neighbors) <= num_neighbors:
            sampled_indices = unique_neighbors; sampled_weights = importance_probs
        else:
            sampled_indices = np.random.choice(unique_neighbors, size=num_neighbors, replace=False, p=importance_probs)
            sampled_weights = np.array([visit_counts[i] for i in sampled_indices], dtype=float); sampled_weights /= sampled_weights.sum()
        new_rows.extend([node_idx] * len(sampled_indices)); new_cols.extend(sampled_indices); new_data.extend(sampled_weights)

    indices = torch.LongTensor([new_rows, new_cols]); values = torch.FloatTensor(new_data)
    return torch.sparse_coo_tensor(indices, values, (total_nodes, total_nodes)).to(device)

# ----------------------------
# 4. Baseline Temporal Training Loop
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
prev_model_state = None
prev_maps = {'u': None, 'i': None, 'e': None}

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    kg_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv").rename(columns={'Src': 'head', 'Dst': 'tail'})
    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    u_map = {u: i for i, u in enumerate(train_df['UserID'].unique())}
    i_map = {m: i for i, m in enumerate(train_df['MovieID'].unique())}
    kg_entities = set(kg_df['head']) | set(kg_df['tail'])
    remaining_entities = [e for e in kg_entities if e not in i_map]
    ent_map = {e: i + len(i_map) for i, e in enumerate(remaining_entities)}
    
    num_u, num_i, num_e = len(u_map), len(i_map), len(ent_map)
    num_nodes = num_u + num_i + num_e
    
    adj = build_adjacency(train_df['UserID'].map(u_map).values, train_df['MovieID'].map(i_map).values, kg_df, u_map, i_map, ent_map)
    model = PinSage(num_nodes).to(device)
    
    # --- FIXED TEMPORAL TRANSFER WITH SAFETY GUARDS ---
    if prev_model_state:
        with torch.no_grad():
            val_device = prev_model_state["emb.weight"].to(device)
            old_u_len = len(prev_maps['u'])
            old_i_len = len(prev_maps['i'])

            # Transfer Users
            for uid, curr_idx in u_map.items():
                if uid in prev_maps['u']:
                    prev_idx = prev_maps['u'][uid]
                    if prev_idx < val_device.shape[0] and curr_idx < num_nodes:
                        model.emb.weight.data[curr_idx] = val_device[prev_idx]

            # Transfer Items
            for mid, curr_idx in i_map.items():
                if mid in prev_maps['i']:
                    prev_idx = prev_maps['i'][mid] + old_u_len
                    target_idx = curr_idx + num_u
                    if prev_idx < val_device.shape[0] and target_idx < num_nodes:
                        model.emb.weight.data[target_idx] = val_device[prev_idx]

            # Transfer Entities
            for eid, curr_idx in ent_map.items():
                if eid in prev_maps['e']:
                    prev_idx = prev_maps['e'][eid] + old_u_len + old_i_len
                    target_idx = curr_idx + num_u + num_i
                    if prev_idx < val_device.shape[0] and target_idx < num_nodes:
                        model.emb.weight.data[target_idx] = val_device[prev_idx]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    model.train()
    for epoch in range(EPOCHS):
        neg_all = torch.randint(0, num_i, (num_train_samples,), device=device)
        idxs = torch.randperm(num_train_samples, device=device)
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_all[b]
            final_emb = model(adj)
            pos = (final_emb[u_b] * final_emb[p_b + num_u]).sum(1)
            neg = (final_emb[u_b] * final_emb[n_b + num_u]).sum(1)
            loss = -torch.nn.functional.logsigmoid(pos - neg).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    
    metrics, count = compute_metrics(model, adj, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"Popularity exposure (percent): {metrics['PopExposure']}\n")
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_maps = {'u': u_map, 'i': i_map, 'e': ent_map}

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0171    | 0.0584    | 0.0116    | 0.0668    | 0.8897
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '4.68%', 'high': '95.32%'}, 'niche': {'low': '1.43%', 'medium': '13.27%', 'high': '85.31%'}}

2 -> 3        | 117    | 0.0074    | 0.0568    | 0.0053    | 0.1424    | 0.8723
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '8.37%', 'high': '91.63%'}, 'niche': {'low': '2.30%', 'medium': '27.43%', 'high': '70.27%'}}

3 -> 4        | 166    | 0.0091    | 0.0581    | 0.0058    | 0.1242    | 0.8628
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '8.39%', 'high': '91.61%'}, 'niche': {'low': '4.71%', 'medium': '22.31%', 'high': '72.98%'}}

4 -> 5        | 211    | 0.0237    | 0.0497    | 0.0218 

K = 25

In [25]:
import os
# CRITICAL FIX: This must be set BEFORE importing torch
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (STAY UNCHANGED)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=25):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj)
        num_users = len(user_map)
        final_u = final_emb[:num_users]
        final_i = final_emb[num_users : num_users + len(item_map)]
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        scores = torch.matmul(final_u[test_u_tensor], final_i.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                seen_nodes = [item_map[m] for m in train_interactions[u_id] if m in item_map]
                if seen_nodes:
                    scores[idx, seen_nodes] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        item_counts = np.zeros(len(item_map))
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            recs = top_indices[idx]
            hits = len(set(recs) & truth)
            recall_arr.append(hits / len(truth))
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(recs):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for r in recs:
                item_counts[r] += 1
                tier = movie_tiers.get(node_to_m_id[r], 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) + abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) + abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(item_counts)
        
        return {'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp}, len(test_u_nodes)

# ----------------------------
# 3. Model Logic (PinSage)
# ----------------------------
class PinSage(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, n_layers=2):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.emb.weight)
        self.layers = nn.ModuleList([nn.Linear(embedding_dim * 2, embedding_dim) for _ in range(n_layers)])
        self.activation = nn.ReLU()

    def forward(self, adj):
        h = self.emb.weight
        for layer in self.layers:
            neighbor_h = torch.sparse.mm(adj, h)
            h_combined = torch.cat([h, neighbor_h], dim=1)
            h = self.activation(layer(h_combined))
            h = nn.functional.normalize(h, p=2, dim=1)
        return h

def build_adjacency(u_list, i_list, kg_df, u_map, i_map, ent_map, num_neighbors=30, num_walks=20, walk_length=5):
    n_u, n_i, n_e = len(u_map), len(i_map), len(ent_map)
    total_nodes = n_u + n_i + n_e
    rows = np.concatenate([u_list, i_list + n_u])
    cols = np.concatenate([i_list + n_u, u_list])
    
    if not kg_df.empty:
        h_nodes = kg_df['head'].map({**i_map, **ent_map}).values
        t_nodes = kg_df['tail'].map({**i_map, **ent_map}).values
        mask = ~np.isnan(h_nodes) & ~np.isnan(t_nodes)
        h_idx = h_nodes[mask].astype(int) + n_u
        t_idx = t_nodes[mask].astype(int) + n_u
        rows = np.concatenate([rows, h_idx, t_idx])
        cols = np.concatenate([cols, t_idx, h_idx])

    adj_mat = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(total_nodes, total_nodes))
    new_rows, new_cols, new_data = [], [], []

    for node_idx in range(total_nodes):
        visit_counts = {}
        for _ in range(num_walks):
            curr = node_idx
            for _ in range(walk_length):
                neighbors = adj_mat.indices[adj_mat.indptr[curr]:adj_mat.indptr[curr+1]]
                if len(neighbors) == 0: break
                curr = np.random.choice(neighbors)
                visit_counts[curr] = visit_counts.get(curr, 0) + 1
        if not visit_counts: continue
        unique_neighbors = list(visit_counts.keys()); counts = np.array(list(visit_counts.values()), dtype=float)
        importance_probs = counts / counts.sum()
        if len(unique_neighbors) <= num_neighbors:
            sampled_indices = unique_neighbors; sampled_weights = importance_probs
        else:
            sampled_indices = np.random.choice(unique_neighbors, size=num_neighbors, replace=False, p=importance_probs)
            sampled_weights = np.array([visit_counts[i] for i in sampled_indices], dtype=float); sampled_weights /= sampled_weights.sum()
        new_rows.extend([node_idx] * len(sampled_indices)); new_cols.extend(sampled_indices); new_data.extend(sampled_weights)

    indices = torch.LongTensor([new_rows, new_cols]); values = torch.FloatTensor(new_data)
    return torch.sparse_coo_tensor(indices, values, (total_nodes, total_nodes)).to(device)

# ----------------------------
# 4. Baseline Temporal Training Loop
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
prev_model_state = None
prev_maps = {'u': None, 'i': None, 'e': None}

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    kg_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv").rename(columns={'Src': 'head', 'Dst': 'tail'})
    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    u_map = {u: i for i, u in enumerate(train_df['UserID'].unique())}
    i_map = {m: i for i, m in enumerate(train_df['MovieID'].unique())}
    kg_entities = set(kg_df['head']) | set(kg_df['tail'])
    remaining_entities = [e for e in kg_entities if e not in i_map]
    ent_map = {e: i + len(i_map) for i, e in enumerate(remaining_entities)}
    
    num_u, num_i, num_e = len(u_map), len(i_map), len(ent_map)
    num_nodes = num_u + num_i + num_e
    
    adj = build_adjacency(train_df['UserID'].map(u_map).values, train_df['MovieID'].map(i_map).values, kg_df, u_map, i_map, ent_map)
    model = PinSage(num_nodes).to(device)
    
    # --- FIXED TEMPORAL TRANSFER WITH SAFETY GUARDS ---
    if prev_model_state:
        with torch.no_grad():
            val_device = prev_model_state["emb.weight"].to(device)
            old_u_len = len(prev_maps['u'])
            old_i_len = len(prev_maps['i'])

            # Transfer Users
            for uid, curr_idx in u_map.items():
                if uid in prev_maps['u']:
                    prev_idx = prev_maps['u'][uid]
                    if prev_idx < val_device.shape[0] and curr_idx < num_nodes:
                        model.emb.weight.data[curr_idx] = val_device[prev_idx]

            # Transfer Items
            for mid, curr_idx in i_map.items():
                if mid in prev_maps['i']:
                    prev_idx = prev_maps['i'][mid] + old_u_len
                    target_idx = curr_idx + num_u
                    if prev_idx < val_device.shape[0] and target_idx < num_nodes:
                        model.emb.weight.data[target_idx] = val_device[prev_idx]

            # Transfer Entities
            for eid, curr_idx in ent_map.items():
                if eid in prev_maps['e']:
                    prev_idx = prev_maps['e'][eid] + old_u_len + old_i_len
                    target_idx = curr_idx + num_u + num_i
                    if prev_idx < val_device.shape[0] and target_idx < num_nodes:
                        model.emb.weight.data[target_idx] = val_device[prev_idx]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    model.train()
    for epoch in range(EPOCHS):
        neg_all = torch.randint(0, num_i, (num_train_samples,), device=device)
        idxs = torch.randperm(num_train_samples, device=device)
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_all[b]
            final_emb = model(adj)
            pos = (final_emb[u_b] * final_emb[p_b + num_u]).sum(1)
            neg = (final_emb[u_b] * final_emb[n_b + num_u]).sum(1)
            loss = -torch.nn.functional.logsigmoid(pos - neg).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    
    metrics, count = compute_metrics(model, adj, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"Popularity exposure (percent): {metrics['PopExposure']}\n")
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_maps = {'u': u_map, 'i': i_map, 'e': ent_map}

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0449    | 0.0778    | 0.0070    | 0.0844    | 0.8836
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '7.83%', 'high': '92.17%'}, 'niche': {'low': '1.80%', 'medium': '18.69%', 'high': '79.51%'}}

2 -> 3        | 117    | 0.0502    | 0.0737    | 0.0074    | 0.1568    | 0.8335
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '3.72%', 'high': '96.28%'}, 'niche': {'low': '1.95%', 'medium': '25.30%', 'high': '72.76%'}}

3 -> 4        | 166    | 0.0303    | 0.0618    | 0.0356    | 0.1195    | 0.8381
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '4.58%', 'high': '95.42%'}, 'niche': {'low': '5.38%', 'medium': '17.12%', 'high': '77.50%'}}

4 -> 5        | 211    | 0.0506    | 0.0703    | 0.0003 

K = 50

In [26]:
import os
# CRITICAL FIX: This must be set BEFORE importing torch
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (STAY UNCHANGED)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=50):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj)
        num_users = len(user_map)
        final_u = final_emb[:num_users]
        final_i = final_emb[num_users : num_users + len(item_map)]
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        scores = torch.matmul(final_u[test_u_tensor], final_i.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                seen_nodes = [item_map[m] for m in train_interactions[u_id] if m in item_map]
                if seen_nodes:
                    scores[idx, seen_nodes] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        item_counts = np.zeros(len(item_map))
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            recs = top_indices[idx]
            hits = len(set(recs) & truth)
            recall_arr.append(hits / len(truth))
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(recs):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for r in recs:
                item_counts[r] += 1
                tier = movie_tiers.get(node_to_m_id[r], 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) + abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) + abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(item_counts)
        
        return {'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp}, len(test_u_nodes)

# ----------------------------
# 3. Model Logic (PinSage)
# ----------------------------
class PinSage(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, n_layers=2):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.emb.weight)
        self.layers = nn.ModuleList([nn.Linear(embedding_dim * 2, embedding_dim) for _ in range(n_layers)])
        self.activation = nn.ReLU()

    def forward(self, adj):
        h = self.emb.weight
        for layer in self.layers:
            neighbor_h = torch.sparse.mm(adj, h)
            h_combined = torch.cat([h, neighbor_h], dim=1)
            h = self.activation(layer(h_combined))
            h = nn.functional.normalize(h, p=2, dim=1)
        return h

def build_adjacency(u_list, i_list, kg_df, u_map, i_map, ent_map, num_neighbors=30, num_walks=20, walk_length=5):
    n_u, n_i, n_e = len(u_map), len(i_map), len(ent_map)
    total_nodes = n_u + n_i + n_e
    rows = np.concatenate([u_list, i_list + n_u])
    cols = np.concatenate([i_list + n_u, u_list])
    
    if not kg_df.empty:
        h_nodes = kg_df['head'].map({**i_map, **ent_map}).values
        t_nodes = kg_df['tail'].map({**i_map, **ent_map}).values
        mask = ~np.isnan(h_nodes) & ~np.isnan(t_nodes)
        h_idx = h_nodes[mask].astype(int) + n_u
        t_idx = t_nodes[mask].astype(int) + n_u
        rows = np.concatenate([rows, h_idx, t_idx])
        cols = np.concatenate([cols, t_idx, h_idx])

    adj_mat = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(total_nodes, total_nodes))
    new_rows, new_cols, new_data = [], [], []

    for node_idx in range(total_nodes):
        visit_counts = {}
        for _ in range(num_walks):
            curr = node_idx
            for _ in range(walk_length):
                neighbors = adj_mat.indices[adj_mat.indptr[curr]:adj_mat.indptr[curr+1]]
                if len(neighbors) == 0: break
                curr = np.random.choice(neighbors)
                visit_counts[curr] = visit_counts.get(curr, 0) + 1
        if not visit_counts: continue
        unique_neighbors = list(visit_counts.keys()); counts = np.array(list(visit_counts.values()), dtype=float)
        importance_probs = counts / counts.sum()
        if len(unique_neighbors) <= num_neighbors:
            sampled_indices = unique_neighbors; sampled_weights = importance_probs
        else:
            sampled_indices = np.random.choice(unique_neighbors, size=num_neighbors, replace=False, p=importance_probs)
            sampled_weights = np.array([visit_counts[i] for i in sampled_indices], dtype=float); sampled_weights /= sampled_weights.sum()
        new_rows.extend([node_idx] * len(sampled_indices)); new_cols.extend(sampled_indices); new_data.extend(sampled_weights)

    indices = torch.LongTensor([new_rows, new_cols]); values = torch.FloatTensor(new_data)
    return torch.sparse_coo_tensor(indices, values, (total_nodes, total_nodes)).to(device)

# ----------------------------
# 4. Baseline Temporal Training Loop
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
prev_model_state = None
prev_maps = {'u': None, 'i': None, 'e': None}

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    kg_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv").rename(columns={'Src': 'head', 'Dst': 'tail'})
    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    u_map = {u: i for i, u in enumerate(train_df['UserID'].unique())}
    i_map = {m: i for i, m in enumerate(train_df['MovieID'].unique())}
    kg_entities = set(kg_df['head']) | set(kg_df['tail'])
    remaining_entities = [e for e in kg_entities if e not in i_map]
    ent_map = {e: i + len(i_map) for i, e in enumerate(remaining_entities)}
    
    num_u, num_i, num_e = len(u_map), len(i_map), len(ent_map)
    num_nodes = num_u + num_i + num_e
    
    adj = build_adjacency(train_df['UserID'].map(u_map).values, train_df['MovieID'].map(i_map).values, kg_df, u_map, i_map, ent_map)
    model = PinSage(num_nodes).to(device)
    
    # --- FIXED TEMPORAL TRANSFER WITH SAFETY GUARDS ---
    if prev_model_state:
        with torch.no_grad():
            val_device = prev_model_state["emb.weight"].to(device)
            old_u_len = len(prev_maps['u'])
            old_i_len = len(prev_maps['i'])

            # Transfer Users
            for uid, curr_idx in u_map.items():
                if uid in prev_maps['u']:
                    prev_idx = prev_maps['u'][uid]
                    if prev_idx < val_device.shape[0] and curr_idx < num_nodes:
                        model.emb.weight.data[curr_idx] = val_device[prev_idx]

            # Transfer Items
            for mid, curr_idx in i_map.items():
                if mid in prev_maps['i']:
                    prev_idx = prev_maps['i'][mid] + old_u_len
                    target_idx = curr_idx + num_u
                    if prev_idx < val_device.shape[0] and target_idx < num_nodes:
                        model.emb.weight.data[target_idx] = val_device[prev_idx]

            # Transfer Entities
            for eid, curr_idx in ent_map.items():
                if eid in prev_maps['e']:
                    prev_idx = prev_maps['e'][eid] + old_u_len + old_i_len
                    target_idx = curr_idx + num_u + num_i
                    if prev_idx < val_device.shape[0] and target_idx < num_nodes:
                        model.emb.weight.data[target_idx] = val_device[prev_idx]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    model.train()
    for epoch in range(EPOCHS):
        neg_all = torch.randint(0, num_i, (num_train_samples,), device=device)
        idxs = torch.randperm(num_train_samples, device=device)
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_all[b]
            final_emb = model(adj)
            pos = (final_emb[u_b] * final_emb[p_b + num_u]).sum(1)
            neg = (final_emb[u_b] * final_emb[n_b + num_u]).sum(1)
            loss = -torch.nn.functional.logsigmoid(pos - neg).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    
    metrics, count = compute_metrics(model, adj, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"Popularity exposure (percent): {metrics['PopExposure']}\n")
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_maps = {'u': u_map, 'i': i_map, 'e': ent_map}

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0853    | 0.0879    | 0.0104    | 0.1073    | 0.8447
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '5.70%', 'high': '94.30%'}, 'niche': {'low': '1.39%', 'medium': '20.41%', 'high': '78.20%'}}

2 -> 3        | 117    | 0.0699    | 0.0750    | 0.0295    | 0.1150    | 0.8165
Popularity exposure (percent): {'mainstream': {'low': '0.09%', 'medium': '7.49%', 'high': '92.42%'}, 'niche': {'low': '2.57%', 'medium': '22.27%', 'high': '75.16%'}}

3 -> 4        | 166    | 0.0746    | 0.0694    | 0.0541    | 0.1157    | 0.8259
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '3.77%', 'high': '96.23%'}, 'niche': {'low': '3.31%', 'medium': '17.83%', 'high': '78.87%'}}

4 -> 5        | 211    | 0.0888    | 0.0764    | 0.0281 

K = 75

In [27]:
import os
# CRITICAL FIX: This must be set BEFORE importing torch
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (STAY UNCHANGED)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=75):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj)
        num_users = len(user_map)
        final_u = final_emb[:num_users]
        final_i = final_emb[num_users : num_users + len(item_map)]
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        scores = torch.matmul(final_u[test_u_tensor], final_i.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                seen_nodes = [item_map[m] for m in train_interactions[u_id] if m in item_map]
                if seen_nodes:
                    scores[idx, seen_nodes] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        item_counts = np.zeros(len(item_map))
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            recs = top_indices[idx]
            hits = len(set(recs) & truth)
            recall_arr.append(hits / len(truth))
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(recs):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for r in recs:
                item_counts[r] += 1
                tier = movie_tiers.get(node_to_m_id[r], 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) + abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) + abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(item_counts)
        
        return {'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp}, len(test_u_nodes)

# ----------------------------
# 3. Model Logic (PinSage)
# ----------------------------
class PinSage(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, n_layers=2):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.emb.weight)
        self.layers = nn.ModuleList([nn.Linear(embedding_dim * 2, embedding_dim) for _ in range(n_layers)])
        self.activation = nn.ReLU()

    def forward(self, adj):
        h = self.emb.weight
        for layer in self.layers:
            neighbor_h = torch.sparse.mm(adj, h)
            h_combined = torch.cat([h, neighbor_h], dim=1)
            h = self.activation(layer(h_combined))
            h = nn.functional.normalize(h, p=2, dim=1)
        return h

def build_adjacency(u_list, i_list, kg_df, u_map, i_map, ent_map, num_neighbors=30, num_walks=20, walk_length=5):
    n_u, n_i, n_e = len(u_map), len(i_map), len(ent_map)
    total_nodes = n_u + n_i + n_e
    rows = np.concatenate([u_list, i_list + n_u])
    cols = np.concatenate([i_list + n_u, u_list])
    
    if not kg_df.empty:
        h_nodes = kg_df['head'].map({**i_map, **ent_map}).values
        t_nodes = kg_df['tail'].map({**i_map, **ent_map}).values
        mask = ~np.isnan(h_nodes) & ~np.isnan(t_nodes)
        h_idx = h_nodes[mask].astype(int) + n_u
        t_idx = t_nodes[mask].astype(int) + n_u
        rows = np.concatenate([rows, h_idx, t_idx])
        cols = np.concatenate([cols, t_idx, h_idx])

    adj_mat = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(total_nodes, total_nodes))
    new_rows, new_cols, new_data = [], [], []

    for node_idx in range(total_nodes):
        visit_counts = {}
        for _ in range(num_walks):
            curr = node_idx
            for _ in range(walk_length):
                neighbors = adj_mat.indices[adj_mat.indptr[curr]:adj_mat.indptr[curr+1]]
                if len(neighbors) == 0: break
                curr = np.random.choice(neighbors)
                visit_counts[curr] = visit_counts.get(curr, 0) + 1
        if not visit_counts: continue
        unique_neighbors = list(visit_counts.keys()); counts = np.array(list(visit_counts.values()), dtype=float)
        importance_probs = counts / counts.sum()
        if len(unique_neighbors) <= num_neighbors:
            sampled_indices = unique_neighbors; sampled_weights = importance_probs
        else:
            sampled_indices = np.random.choice(unique_neighbors, size=num_neighbors, replace=False, p=importance_probs)
            sampled_weights = np.array([visit_counts[i] for i in sampled_indices], dtype=float); sampled_weights /= sampled_weights.sum()
        new_rows.extend([node_idx] * len(sampled_indices)); new_cols.extend(sampled_indices); new_data.extend(sampled_weights)

    indices = torch.LongTensor([new_rows, new_cols]); values = torch.FloatTensor(new_data)
    return torch.sparse_coo_tensor(indices, values, (total_nodes, total_nodes)).to(device)

# ----------------------------
# 4. Baseline Temporal Training Loop
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
prev_model_state = None
prev_maps = {'u': None, 'i': None, 'e': None}

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    kg_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv").rename(columns={'Src': 'head', 'Dst': 'tail'})
    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    u_map = {u: i for i, u in enumerate(train_df['UserID'].unique())}
    i_map = {m: i for i, m in enumerate(train_df['MovieID'].unique())}
    kg_entities = set(kg_df['head']) | set(kg_df['tail'])
    remaining_entities = [e for e in kg_entities if e not in i_map]
    ent_map = {e: i + len(i_map) for i, e in enumerate(remaining_entities)}
    
    num_u, num_i, num_e = len(u_map), len(i_map), len(ent_map)
    num_nodes = num_u + num_i + num_e
    
    adj = build_adjacency(train_df['UserID'].map(u_map).values, train_df['MovieID'].map(i_map).values, kg_df, u_map, i_map, ent_map)
    model = PinSage(num_nodes).to(device)
    
    # --- FIXED TEMPORAL TRANSFER WITH SAFETY GUARDS ---
    if prev_model_state:
        with torch.no_grad():
            val_device = prev_model_state["emb.weight"].to(device)
            old_u_len = len(prev_maps['u'])
            old_i_len = len(prev_maps['i'])

            # Transfer Users
            for uid, curr_idx in u_map.items():
                if uid in prev_maps['u']:
                    prev_idx = prev_maps['u'][uid]
                    if prev_idx < val_device.shape[0] and curr_idx < num_nodes:
                        model.emb.weight.data[curr_idx] = val_device[prev_idx]

            # Transfer Items
            for mid, curr_idx in i_map.items():
                if mid in prev_maps['i']:
                    prev_idx = prev_maps['i'][mid] + old_u_len
                    target_idx = curr_idx + num_u
                    if prev_idx < val_device.shape[0] and target_idx < num_nodes:
                        model.emb.weight.data[target_idx] = val_device[prev_idx]

            # Transfer Entities
            for eid, curr_idx in ent_map.items():
                if eid in prev_maps['e']:
                    prev_idx = prev_maps['e'][eid] + old_u_len + old_i_len
                    target_idx = curr_idx + num_u + num_i
                    if prev_idx < val_device.shape[0] and target_idx < num_nodes:
                        model.emb.weight.data[target_idx] = val_device[prev_idx]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    model.train()
    for epoch in range(EPOCHS):
        neg_all = torch.randint(0, num_i, (num_train_samples,), device=device)
        idxs = torch.randperm(num_train_samples, device=device)
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_all[b]
            final_emb = model(adj)
            pos = (final_emb[u_b] * final_emb[p_b + num_u]).sum(1)
            neg = (final_emb[u_b] * final_emb[n_b + num_u]).sum(1)
            loss = -torch.nn.functional.logsigmoid(pos - neg).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    
    metrics, count = compute_metrics(model, adj, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"Popularity exposure (percent): {metrics['PopExposure']}\n")
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_maps = {'u': u_map, 'i': i_map, 'e': ent_map}

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.1324    | 0.1072    | 0.0375    | 0.0944    | 0.8339
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '5.22%', 'high': '94.78%'}, 'niche': {'low': '2.50%', 'medium': '16.87%', 'high': '80.63%'}}

2 -> 3        | 117    | 0.1050    | 0.0871    | 0.0783    | 0.1144    | 0.7990
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '4.90%', 'high': '95.10%'}, 'niche': {'low': '1.91%', 'medium': '20.14%', 'high': '77.95%'}}

3 -> 4        | 166    | 0.1018    | 0.0788    | 0.0396    | 0.1230    | 0.8154
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '3.14%', 'high': '96.86%'}, 'niche': {'low': '4.23%', 'medium': '17.36%', 'high': '78.41%'}}

4 -> 5        | 211    | 0.1158    | 0.0803    | 0.0690 

K = 100

In [28]:
import os
# CRITICAL FIX: This must be set BEFORE importing torch
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (STAY UNCHANGED)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=100):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj)
        num_users = len(user_map)
        final_u = final_emb[:num_users]
        final_i = final_emb[num_users : num_users + len(item_map)]
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        scores = torch.matmul(final_u[test_u_tensor], final_i.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                seen_nodes = [item_map[m] for m in train_interactions[u_id] if m in item_map]
                if seen_nodes:
                    scores[idx, seen_nodes] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        item_counts = np.zeros(len(item_map))
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            recs = top_indices[idx]
            hits = len(set(recs) & truth)
            recall_arr.append(hits / len(truth))
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(recs):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for r in recs:
                item_counts[r] += 1
                tier = movie_tiers.get(node_to_m_id[r], 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) + abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) + abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(item_counts)
        
        return {'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp}, len(test_u_nodes)

# ----------------------------
# 3. Model Logic (PinSage)
# ----------------------------
class PinSage(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, n_layers=2):
        super().__init__()
        self.n_layers = n_layers
        self.emb = nn.Embedding(num_nodes, embedding_dim)
        nn.init.xavier_uniform_(self.emb.weight)
        self.layers = nn.ModuleList([nn.Linear(embedding_dim * 2, embedding_dim) for _ in range(n_layers)])
        self.activation = nn.ReLU()

    def forward(self, adj):
        h = self.emb.weight
        for layer in self.layers:
            neighbor_h = torch.sparse.mm(adj, h)
            h_combined = torch.cat([h, neighbor_h], dim=1)
            h = self.activation(layer(h_combined))
            h = nn.functional.normalize(h, p=2, dim=1)
        return h

def build_adjacency(u_list, i_list, kg_df, u_map, i_map, ent_map, num_neighbors=30, num_walks=20, walk_length=5):
    n_u, n_i, n_e = len(u_map), len(i_map), len(ent_map)
    total_nodes = n_u + n_i + n_e
    rows = np.concatenate([u_list, i_list + n_u])
    cols = np.concatenate([i_list + n_u, u_list])
    
    if not kg_df.empty:
        h_nodes = kg_df['head'].map({**i_map, **ent_map}).values
        t_nodes = kg_df['tail'].map({**i_map, **ent_map}).values
        mask = ~np.isnan(h_nodes) & ~np.isnan(t_nodes)
        h_idx = h_nodes[mask].astype(int) + n_u
        t_idx = t_nodes[mask].astype(int) + n_u
        rows = np.concatenate([rows, h_idx, t_idx])
        cols = np.concatenate([cols, t_idx, h_idx])

    adj_mat = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(total_nodes, total_nodes))
    new_rows, new_cols, new_data = [], [], []

    for node_idx in range(total_nodes):
        visit_counts = {}
        for _ in range(num_walks):
            curr = node_idx
            for _ in range(walk_length):
                neighbors = adj_mat.indices[adj_mat.indptr[curr]:adj_mat.indptr[curr+1]]
                if len(neighbors) == 0: break
                curr = np.random.choice(neighbors)
                visit_counts[curr] = visit_counts.get(curr, 0) + 1
        if not visit_counts: continue
        unique_neighbors = list(visit_counts.keys()); counts = np.array(list(visit_counts.values()), dtype=float)
        importance_probs = counts / counts.sum()
        if len(unique_neighbors) <= num_neighbors:
            sampled_indices = unique_neighbors; sampled_weights = importance_probs
        else:
            sampled_indices = np.random.choice(unique_neighbors, size=num_neighbors, replace=False, p=importance_probs)
            sampled_weights = np.array([visit_counts[i] for i in sampled_indices], dtype=float); sampled_weights /= sampled_weights.sum()
        new_rows.extend([node_idx] * len(sampled_indices)); new_cols.extend(sampled_indices); new_data.extend(sampled_weights)

    indices = torch.LongTensor([new_rows, new_cols]); values = torch.FloatTensor(new_data)
    return torch.sparse_coo_tensor(indices, values, (total_nodes, total_nodes)).to(device)

# ----------------------------
# 4. Baseline Temporal Training Loop
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
prev_model_state = None
prev_maps = {'u': None, 'i': None, 'e': None}

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())

for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    kg_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv").rename(columns={'Src': 'head', 'Dst': 'tail'})
    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    u_map = {u: i for i, u in enumerate(train_df['UserID'].unique())}
    i_map = {m: i for i, m in enumerate(train_df['MovieID'].unique())}
    kg_entities = set(kg_df['head']) | set(kg_df['tail'])
    remaining_entities = [e for e in kg_entities if e not in i_map]
    ent_map = {e: i + len(i_map) for i, e in enumerate(remaining_entities)}
    
    num_u, num_i, num_e = len(u_map), len(i_map), len(ent_map)
    num_nodes = num_u + num_i + num_e
    
    adj = build_adjacency(train_df['UserID'].map(u_map).values, train_df['MovieID'].map(i_map).values, kg_df, u_map, i_map, ent_map)
    model = PinSage(num_nodes).to(device)
    
    # --- FIXED TEMPORAL TRANSFER WITH SAFETY GUARDS ---
    if prev_model_state:
        with torch.no_grad():
            val_device = prev_model_state["emb.weight"].to(device)
            old_u_len = len(prev_maps['u'])
            old_i_len = len(prev_maps['i'])

            # Transfer Users
            for uid, curr_idx in u_map.items():
                if uid in prev_maps['u']:
                    prev_idx = prev_maps['u'][uid]
                    if prev_idx < val_device.shape[0] and curr_idx < num_nodes:
                        model.emb.weight.data[curr_idx] = val_device[prev_idx]

            # Transfer Items
            for mid, curr_idx in i_map.items():
                if mid in prev_maps['i']:
                    prev_idx = prev_maps['i'][mid] + old_u_len
                    target_idx = curr_idx + num_u
                    if prev_idx < val_device.shape[0] and target_idx < num_nodes:
                        model.emb.weight.data[target_idx] = val_device[prev_idx]

            # Transfer Entities
            for eid, curr_idx in ent_map.items():
                if eid in prev_maps['e']:
                    prev_idx = prev_maps['e'][eid] + old_u_len + old_i_len
                    target_idx = curr_idx + num_u + num_i
                    if prev_idx < val_device.shape[0] and target_idx < num_nodes:
                        model.emb.weight.data[target_idx] = val_device[prev_idx]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    model.train()
    for epoch in range(EPOCHS):
        neg_all = torch.randint(0, num_i, (num_train_samples,), device=device)
        idxs = torch.randperm(num_train_samples, device=device)
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_all[b]
            final_emb = model(adj)
            pos = (final_emb[u_b] * final_emb[p_b + num_u]).sum(1)
            neg = (final_emb[u_b] * final_emb[n_b + num_u]).sum(1)
            loss = -torch.nn.functional.logsigmoid(pos - neg).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    
    metrics, count = compute_metrics(model, adj, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"Popularity exposure (percent): {metrics['PopExposure']}\n")
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_maps = {'u': u_map, 'i': i_map, 'e': ent_map}

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.1550    | 0.1127    | 0.0191    | 0.0857    | 0.8330
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '6.38%', 'high': '93.62%'}, 'niche': {'low': '1.96%', 'medium': '17.29%', 'high': '80.76%'}}

2 -> 3        | 117    | 0.1364    | 0.1010    | 0.0598    | 0.1520    | 0.7855
Popularity exposure (percent): {'mainstream': {'low': '0.00%', 'medium': '5.35%', 'high': '94.65%'}, 'niche': {'low': '4.66%', 'medium': '23.49%', 'high': '71.85%'}}

3 -> 4        | 166    | 0.1384    | 0.0937    | 0.0740    | 0.1288    | 0.8109
Popularity exposure (percent): {'mainstream': {'low': '0.03%', 'medium': '6.29%', 'high': '93.68%'}, 'niche': {'low': '2.83%', 'medium': '22.82%', 'high': '74.36%'}}

4 -> 5        | 211    | 0.1601    | 0.0996    | 0.0546 